Here is the code with the commentary about improvements and fixes removed.

### Cell 1: Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')
from transformers import AutoTokenizer
import tf_keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, TFBertModel
from transformers import TFBertForSequenceClassification
import tensorflow as tf
import os
import torch

# Create checkpoints directory
checkpoint_dir = 'checkpoints'
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
    print(f"Directory '{checkpoint_dir}' created successfully!")
else:
    print(f"Directory '{checkpoint_dir}' already exists.")

### Cell 2: Data Loading & Downsampling

In [ ]:
# Upload dataset
try:
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
except Exception:
    file_path = input("enter path: ").strip()

df = pd.read_csv(file_path)

df = df.sample(frac=0.1, random_state=42).reset_index(drop=True)
print(f"Dataset downsampled to 10%. New shape: {df.shape}")

# Shard the data (split the data) ~ 10 shards, 10 splits
# Note: If data is very small, we reduce shards to avoid empty splits
num_shards = 10
if len(df) < 100:
    num_shards = 2
    print(f"Dataset too small for 10 shards, reducing to {num_shards} shards.")

parts = np.array_split(df.sample(frac=1, random_state=42).reset_index(drop=True), num_shards)

print(f"Data split into {len(parts)} shards.")
print(df.shape)

### Cell 3: Preprocessing & Tokenization

In [ ]:

def make_text(row):
    return (
        f"Patient Profile: Age {row['Age']}, {row['Gender']}, Blood Type {row['Blood Type']}. "
        f"Medical Condition: {row['Medical Condition']}. "
        f"Admission Date: {row['Date of Admission']}, Discharge Date: {row['Discharge Date']}. "
        f"Prescribed Medication: {row['Medication']}. "
        f"Test Result Classification:"
    )

df = df.fillna("Unknown")
df['text_input'] = df.apply(make_text, axis=1)

# create labels(y)
label_map = {
    "Normal": 0,
    "Abnormal": 1,
    "Inconclusive": 2
}
df['label'] = df["Test Results"].map(label_map)
df['label'] = df['label'].astype(int)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
encodings = tokenizer(
    df['text_input'].tolist(),
    truncation=True,
    padding="max_length",
    max_length=32,
    return_tensors="tf"
)

### Cell 4: Dataset Helper Functions

In [ ]:

def create_dataset(indices, batch_size=4, shuffle=False):
    labels_tensor = tf.constant(df['label'].values, dtype=tf.int32)

    dataset_of_indices = tf.data.Dataset.from_tensor_slices(indices)

    def _map_fn(idx):
        input_ids = encodings["input_ids"][idx]
        attention_mask = encodings["attention_mask"][idx]
        label = labels_tensor[idx]
        return {"input_ids": input_ids, "attention_mask": attention_mask}, label

    dataset = dataset_of_indices.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        # Reduced shuffle buffer size for smaller dataset speed
        dataset = dataset.shuffle(min(2000, len(indices) + 1))

    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def split_indices(parts):
    train_plus, val_plus = [], []

    all_indices = df.index.tolist()

    # Stratify might fail if classes are too small in the 10% slice, so we use try/except
    try:
        train_val_indices, test_indices = train_test_split(
            all_indices,
            test_size=0.15,
            random_state=42,
            stratify=df['label']
        )
    except ValueError:
        # Fallback if stratify fails
        print("Warning: Stratification failed due to small dataset size. proceeding without stratification.")
        train_val_indices, test_indices = train_test_split(
            all_indices,
            test_size=0.15,
            random_state=42
        )

    for i in range(len(parts)):
        shard_indices = parts[i].index.tolist()
        shard_train_val = [idx for idx in shard_indices if idx in train_val_indices]

        if len(shard_train_val) > 0:
            try:
                train_idx, val_idx = train_test_split(
                    shard_train_val, test_size=0.15, random_state=42
                )
            except ValueError:
                # Fallback for very small shards
                train_idx = shard_train_val
                val_idx = []

            train_plus.append(train_idx)
            val_plus.append(val_idx)
        else:
            train_plus.append([])
            val_plus.append([])

    return train_plus, val_plus, test_indices


def get_y_true(test_indices):
    return df.loc[test_indices, 'label'].values

### Cell 5: Model Helper Functions

In [ ]:


def create_model():
    model = TFBertForSequenceClassification.from_pretrained(
          "bert-base-uncased",
          num_labels=3,
          use_safetensors=False
          )

    # Better optimizer with learning rate schedule
    lr_schedule = tf_keras.optimizers.schedules.PolynomialDecay(
        initial_learning_rate=3e-5,
        decay_steps=1000,
        end_learning_rate=1e-6
    )

    optimizer = tf_keras.optimizers.Adam(
        learning_rate=lr_schedule,
        epsilon=1e-8,
        clipnorm=1.0  # Gradient clipping
    )

    model.compile(
          optimizer=optimizer,
          loss=tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True),
          metrics=["accuracy"]
          )
    return model


def create_slice_idx(dataset_idx, num_slices=5):
    # Handle cases where dataset is smaller than num_slices
    if len(dataset_idx) < num_slices:
        return np.array_split(dataset_idx, len(dataset_idx))
    return np.array_split(dataset_idx, num_slices)


def train_shard(train_idx, val_idx, shard_no, num_slices=5):
    if len(train_idx) == 0:
        return None

    train_slices = create_slice_idx(train_idx, num_slices)
    real_num_slices = len(train_slices) # Update incase slices were reduced due to size

    model = create_model()

    # Handle empty validation set
    if len(val_idx) > 0:
        val = create_dataset(val_idx, shuffle=False)
        validation_data = val
    else:
        validation_data = None

    for i in range(real_num_slices):
        if len(train_slices[i]) == 0: continue

        train = create_dataset(train_slices[i], shuffle=True)
        checkpoint_path = os.path.join("checkpoints", f"shard_{shard_no}_slice_{i}.h5")

        model.fit(
            train,
            validation_data=validation_data,
            epochs=3,
            verbose=1
        )
        model.save_weights(checkpoint_path)
    return model

### Cell 6: Orchestration & Unlearning Functions

In [ ]:


def train_all_shards():
    train_idx, val_idx, test_idx = split_indices(parts)
    test = create_dataset(test_idx, shuffle=False)
    models = {}

    for i in range(len(parts)):
        if len(train_idx[i]) > 0:
            print(f"\n{'='*50}")
            print(f"Training Shard {i+1}/{len(parts)}")
            print(f"{'='*50}")
            model = train_shard(train_idx[i], val_idx[i], i, 5)
            if model:
                models[i] = model

    return models, test, test_idx


def predict_shard(model, test):
    probs = model.predict(test).logits
    return np.argmax(probs, axis=1)


def aggregate_models(models, test_plus, y_true):
    if not models:
        return 0.0

    # Get prediction from first available model to determine length
    first_model_idx = list(models.keys())[0]
    len_h = predict_shard(models[first_model_idx], test_plus)

    rows = len(models)
    columns = len(len_h)
    result_ls = []

    # Map model indices to matrix rows
    model_indices = list(models.keys())
    result_matrix = np.zeros((rows, columns), dtype=np.int64)

    for i, model_idx in enumerate(model_indices):
        result_matrix[i] = predict_shard(models[model_idx], test_plus)

    for i in range(columns):
        result_ls.append(np.argmax(np.bincount(result_matrix[:, i])))

    accuracy = accuracy_score(y_true, result_ls)
    return accuracy


def unlearn_data(indices, parts, models, num_slices=5):
    # Only keep indices that actually exist in our downsampled df
    valid_indices = [idx for idx in indices if idx in df.index]
    if len(valid_indices) == 0:
        print("None of the unlearning indices were found in the current dataset.")
        return models

    deldata = df.loc[valid_indices]
    shard_indices = {}

    for idx in deldata.index:
        found = False
        for i, shard in enumerate(parts):
            if idx in shard.index:
                if i not in shard_indices:
                    shard_indices[i] = []
                shard_indices[i].append(idx)
                found = True
                break

    if not shard_indices:
        print("No data to unlearn.")
        return models

    train_idx, val_idx, test_idx = split_indices(parts)

    for shard_no, idxs in shard_indices.items():
        if shard_no not in models: continue

        print(f"\nRetraining shard {shard_no} after unlearning...")
        slice_idx = create_slice_idx(train_idx[shard_no], num_slices=5)
        affected_slice = set()

        for slice_no, slice_indices in enumerate(slice_idx):
            slice_indices_list = list(slice_indices)
            for idx in idxs:
                if idx in slice_indices_list:
                    affected_slice.add(slice_no)
                    slice_indices_list.remove(idx)
            slice_idx[slice_no] = np.array(slice_indices_list)

        if not affected_slice:
            continue

        model = models[shard_no]
        first = min(affected_slice)

        if first > 0:
            checkpoint_path = os.path.join("checkpoints", f"shard_{shard_no}_slice_{first-1}.h5")
            if os.path.exists(checkpoint_path):
                model.load_weights(checkpoint_path)

        if len(val_idx[shard_no]) > 0:
            val = create_dataset(val_idx[shard_no], shuffle=False)
            validation_data = val
        else:
            validation_data = None

        for i in range(first, len(slice_idx)):
            if len(slice_idx[i]) == 0:
                continue
            train = create_dataset(slice_idx[i], shuffle=True)
            checkpoint_path = os.path.join("checkpoints", f"shard_{shard_no}_slice_{i}.h5")
            model.fit(
                train,
                validation_data=validation_data,
                epochs=3,
                verbose=1
            )
            model.save_weights(checkpoint_path)
        models[shard_no] = model
    return models


def validate_unlearning(models, test_plus, unlearned_indices, y_true):
    # Filter indices that exist in DF
    valid_unlearned_indices = [idx for idx in unlearned_indices if idx in df.index]

    if not valid_unlearned_indices:
        print("No valid indices to validate unlearning.")
        return {}

    unlearned_data_subset = df.loc[valid_unlearned_indices]

    unlearned_texts = unlearned_data_subset['text_input'].tolist()
    unlearned_encodings = tokenizer(
        unlearned_texts,
        truncation=True,
        padding="max_length",
        max_length=32,
        return_tensors="tf"
    )

    unlearned_dataset = tf.data.Dataset.from_tensor_slices((
        {
            "input_ids": unlearned_encodings["input_ids"],
            "attention_mask": unlearned_encodings["attention_mask"]
        },
        unlearned_data_subset['label'].values
    )).batch(4)

    unlearned_y_true = unlearned_data_subset['label'].values

    # Predict using available models
    model_indices = list(models.keys())
    rows = len(models)
    columns = len(valid_unlearned_indices)
    result_matrix = np.zeros((rows, columns), dtype=np.int64)

    for i, model_idx in enumerate(model_indices):
         result_matrix[i] = predict_shard(models[model_idx], unlearned_dataset)

    result_ls = []
    for i in range(columns):
        result_ls.append(np.argmax(np.bincount(result_matrix[:, i])))

    unlearned_accuracy = accuracy_score(unlearned_y_true, result_ls)
    overall_accuracy = aggregate_models(models, test_plus, y_true)

    validation_results = {
        'unlearned_data_accuracy': unlearned_accuracy,
        'overall_test_accuracy': overall_accuracy,
        'num_unlearned_samples': len(valid_unlearned_indices),
        'unlearned_indices': valid_unlearned_indices
    }

    print(f"\n{'='*50}")
    print(f"  UNLEARNING VALIDATION RESULTS")
    print(f"{'='*50}")
    print(f"Number of unlearned samples: {len(valid_unlearned_indices)}")
    print(f"Accuracy on unlearned data: {unlearned_accuracy:.4f}")
    print(f"Overall test accuracy: {overall_accuracy:.4f}")

    if unlearned_accuracy < 0.4:
        print("✓ Unlearning appears successful (accuracy close to random)")
    else:
        print("⚠ Model may still retain some memory of unlearned data")

    return validation_results

### Cell 7: Execution Main Loop

In [ ]:
print("\n" + "="*50)
print("  STARTING MODEL TRAINING")
print("="*50)

models, test_plus, test_idx = train_all_shards()
y_true = get_y_true(test_idx)

initial_accuracy = aggregate_models(models, test_plus, y_true)
print(f"\n{'='*50}")
print(f"  INITIAL MODEL PERFORMANCE")
print(f"{'='*50}")
print(f"Initial accuracy: {initial_accuracy:.4f}")

np.random.seed(42)

sample_size = min(100, len(df))
unlearn_indices = np.random.choice(df.index, size=sample_size, replace=False)

print(f"\n{'='*50}")
print(f"  PERFORMING UNLEARNING (Samples: {sample_size})")
print(f"{'='*50}")
updated_models = unlearn_data(unlearn_indices, parts, models, num_slices=5)

validation_results = validate_unlearning(updated_models, test_plus, unlearn_indices, y_true)

final_accuracy = aggregate_models(updated_models, test_plus, y_true)
print(f"\n{'='*50}")
print(f"  FINAL MODEL PERFORMANCE")
print(f"{'='*50}")
print(f"Final accuracy: {final_accuracy:.4f}")
print(f"Accuracy change: {final_accuracy - initial_accuracy:+.4f}")
print(f"{'='*50}\n")